# Temporal spectral embedding of volatility residuals

**Research question.** Do realized-volatility (RV) dynamics live on a small
number of recurring *regimes*, and can we recover that regime structure with
an unsupervised manifold-learning method?

**Approach.** For each trading day we form a *view* — the 20-day window of
residuals left over after a HAR-style Ridge baseline has explained the
linear/multi-scale part of RV. We then build a **spectral embedding**
(graph-Laplacian eigenmaps) over these views: each day becomes a point whose
neighbours are days with a similar recent residual trajectory. If regimes
exist, similar market states (e.g. crisis periods) should cluster in the
embedding.

**Pipeline.**

| Step | What | Where |
|---|---|---|
| 1 | Load 30-min RV, build diurnal-adjusted target + HAR/calendar features | `src.backtest.executor.load_and_transform` |
| 2 | Walk-forward Ridge baseline → residuals (the regime signal) | `src.features.transforms.residualizer.Residualizer` |
| 3 | Slide a 20-day window over residuals → one *view* per day | this notebook |
| 4 | Spectral embedding of views (k-NN graph → Laplacian → eigvecs) | `sklearn.manifold.SpectralEmbedding` |
| 5 | Diagnose what the embedding captures | colored scatter plots |

This notebook is also the **source of truth** for the embedding code: the
cells tagged `# export` are auto-exported to
`src/features/extractors/spectral_embedding.py`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO = Path.cwd()
while REPO.parent != REPO and not (REPO / "src").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.linear_model import Ridge

from src.backtest.executor import _build_har_and_calendar, load_and_transform
from src.features.transforms.residualizer import Residualizer
from src.features.transforms.scaling import rolling_robust_scale
from src.features.transforms.target import PERIODS_PER_DAY

## 1. Load and feature-engineer the RV data

30-minute realized-volatility bars, 2005–2024. `load_and_transform` produces
the diurnal-adjusted target `adj_RV` (intraday seasonality removed, 240-period
winsorized) and `_build_har_and_calendar` adds the 13 predictors: six HAR
rolling-means at geometric lags `[1, 5, 25, 125, 625, 3125]` plus calendar
dummies. The target is `adj_RV` shifted by the forecast `horizon`.

In [ ]:
HORIZON = 1

df, _ = load_and_transform(
    "data", exog_cols=[],
    target_use_diurnal=True, target_winsor_window=240, dropna_with_exog=True,
)
df, feature_names = _build_har_and_calendar(df, exog_cols=[], add_calendar=True)
df["target"] = df["adj_RV"].shift(-HORIZON)
df = df.dropna(subset=["target"] + feature_names).reset_index(drop=True)
df["t"] = pd.to_datetime(df["t"])

print(f"rows:             {len(df):,}")
print(f"date range:       {df['t'].min()}  ..  {df['t'].max()}")
print(f"feature columns:  {len(feature_names)}  -> {feature_names}")

## 2. Walk-forward Ridge baseline → residuals

We are interested in the part of RV that a standard HAR model *cannot*
explain — that is where regime structure should hide. `Residualizer`
walk-forwards a Ridge fit (refit on a rolling window) and returns the causal
out-of-sample residual stream. This is the same `Residualizer` the production
`spectral_knn` backtest uses, so the residuals here match what the model sees.

`refit_frequency = 48` refits Ridge once per trading day — fast, and the
baseline barely drifts within a day.

In [ ]:
WARMUP_DAYS = 500
train_win = WARMUP_DAYS * PERIODS_PER_DAY
RIDGE_ALPHA = 1.0
RESIDUAL_REFIT_FREQUENCY = PERIODS_PER_DAY  # refit Ridge daily (48 bars)

X = df[feature_names].to_numpy(dtype=np.float64)
y = df["target"].to_numpy(dtype=np.float64)

X_scaled = rolling_robust_scale(X, train_win)

# Walk-forward residuals: at each step t >= train_win, refit Ridge on
# [t-train_win:t], emit y[t] - ridge.predict(X[t]). Same residualization
# step MultiStageBacktest does, exposed as a one-call API.
res = Residualizer(lambda: Ridge(alpha=RIDGE_ALPHA))
residuals = res.walk_forward_residuals(
    X_scaled, y, train_win,
    refit_frequency=RESIDUAL_REFIT_FREQUENCY,
    progress=True,
)
dates_resid = df["t"].iloc[train_win:].reset_index(drop=True)

print(f"residuals: {residuals.shape}    OOS region: {dates_resid.iloc[0]}  ..  {dates_resid.iloc[-1]}")
print(f"R^2 OOS (walk-forward Ridge on HAR features): {1 - residuals.var() / y[train_win:].var():.4f}")

In [ ]:
# Residual series over time, with major risk-off episodes shaded.
fig, ax = plt.subplots(figsize=(13, 3.0))
ax.plot(dates_resid, residuals, lw=0.3, c="k", alpha=0.6)
ax.axhline(0, c="gray", lw=0.5)
for label, lo, hi, color in [
    ("GFC", "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt", "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue", "2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon", "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID", "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23", "2023-03-08", "2023-03-22", "tab:green"),
]:
    ax.axvspan(pd.Timestamp(lo), pd.Timestamp(hi), alpha=0.18, color=color, label=label)
ax.legend(fontsize=7, ncol=6, loc="upper right")
ax.set_xlabel("date")
ax.set_ylabel("HAR-Ridge residual  (adj_RV units)")
ax.set_ylim(np.quantile(residuals, [0.001, 0.999]))
fig.suptitle(
    "Walk-forward HAR-Ridge residuals, 2006–2024\n"
    "Shaded = known risk-off episodes. Residual variance clusters in crises — "
    "the regime signal we hope the embedding will organize.",
    fontsize=10,
)
plt.tight_layout(); plt.show()

## 3. The spectral-embedding machinery  *(source of truth)*

The three `# export` cells below define the module that ships to
`src/features/extractors/spectral_embedding.py`:

- `build_embedding(views, d, k_graph)` — k-NN affinity graph + normalized
  Laplacian + bottom-`d` eigenvectors, via `sklearn.manifold.SpectralEmbedding`.
- `SpectralBasis` — holds the training embedding `phi_train` and a
  `NearestNeighbors` index; `.embed(v)` extends a *new* view into the
  embedding (Nyström out-of-sample extension, since sklearn ships no
  `.transform()`).

Edit the embedding here, not in `src/` — the src file is regenerated.

In [ ]:
# export
"""Temporal spectral embedding (sklearn-backed).

A thin wrapper over :class:`sklearn.manifold.SpectralEmbedding` that gives
:class:`~src.backtest.multi_stage.MultiStageBacktest` the
``.phi_train + .embed(v_test)`` interface it expects. Out-of-sample
extension is a weighted-kNN Nyström over the training views.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from sklearn.manifold import SpectralEmbedding
from sklearn.neighbors import NearestNeighbors

In [ ]:
# export
@dataclass
class SpectralBasis:
    """Frozen training-side spectral embedding state.

    Holds the training-side embedding ``phi_train`` plus a fitted
    ``NearestNeighbors`` index over the training views (the NN index
    already retains the views internally in ``_fit_X``).
    ``sklearn.manifold.SpectralEmbedding`` doesn't implement
    ``transform()``, so the NN index drives weighted-kNN Nyström
    extension for new test points.
    """

    phi_train: np.ndarray
    k_graph: int
    nn_index: NearestNeighbors

    def embed(self, v_test: np.ndarray) -> np.ndarray:
        """Embed a single test view via weighted-kNN Nyström. Returns shape (d,)."""
        dists, idx = self.nn_index.kneighbors(v_test[None, :], n_neighbors=self.k_graph)
        dists = dists.ravel()
        idx = idx.ravel()
        sigma = float(np.median(dists)) + 1e-12
        w = np.exp(-(dists**2) / (2 * sigma**2))
        s = w.sum()
        if s <= 0:
            return self.phi_train[idx].mean(axis=0)
        w = w / s
        return (w[:, None] * self.phi_train[idx]).sum(axis=0)

    def embed_batch(self, V_test: np.ndarray) -> np.ndarray:
        """Embed a batch of test views. Returns shape (M, d). Vectorized."""
        dists, idx = self.nn_index.kneighbors(V_test, n_neighbors=self.k_graph)
        sigma = np.median(dists, axis=1, keepdims=True) + 1e-12
        w = np.exp(-(dists**2) / (2 * sigma**2))
        w = w / np.clip(w.sum(axis=1, keepdims=True), 1e-12, None)
        return np.einsum("mk,mkd->md", w, self.phi_train[idx])

In [ ]:
# export
def build_embedding(views: np.ndarray, d: int, k_graph: int, seed: int = 42) -> SpectralBasis:
    """Spectral embedding of temporal views.

    Delegates to :class:`sklearn.manifold.SpectralEmbedding` with binary
    ``affinity='nearest_neighbors'`` (k = ``k_graph``) and ARPACK
    eigensolver. The fitted ``NearestNeighbors`` index over ``views`` is
    cached on the returned :class:`SpectralBasis` so out-of-sample test
    points can be embedded via Nyström without re-fitting anything.
    """
    spectral = SpectralEmbedding(
        n_components=d,
        affinity="nearest_neighbors",
        n_neighbors=k_graph,
        random_state=seed,
    )
    phi_train = spectral.fit_transform(views)
    nn_index = NearestNeighbors(n_neighbors=k_graph).fit(views)
    return SpectralBasis(
        phi_train=phi_train,
        k_graph=k_graph,
        nn_index=nn_index,
    )

## 4. Form views and build the embedding

A **view** at day *t* is the `W = 960`-bar (20-trading-day) window of
residuals immediately preceding *t*. We subsample to one view per day
(~4 600 views, 2007–2024) so the eigendecomposition is fast. Each view is one
point fed to the spectral embedding; its neighbours are days with a similar
recent residual trajectory.

In [ ]:
VIEW_WINDOW = 960               # 20 days * 48 bars
SUBSAMPLE_STEP = PERIODS_PER_DAY  # one view per day
EMBEDDING_DIM = 8
GRAPH_K = 10

view_idx = np.arange(VIEW_WINDOW, len(residuals), SUBSAMPLE_STEP)
views = np.stack([residuals[i - VIEW_WINDOW : i] for i in view_idx])
view_dates = dates_resid.iloc[view_idx].reset_index(drop=True)

print(f"views shape: {views.shape}   (N views, W bars each)")
print(f"date span:   {view_dates.iloc[0]}  ..  {view_dates.iloc[-1]}")

In [ ]:
%time basis = build_embedding(views, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)

phi = basis.phi_train
print(f"phi: {phi.shape}")

## 5. What does the embedding capture?

We color the 2-D embedding two ways: by **calendar year** (do regimes track
epochs?) and by **view RMS** (the window's residual amplitude, i.e. a
volatility-level proxy). A useful regime embedding would cluster by market
state, not merely by amplitude.

In [ ]:
view_years = view_dates.dt.year.to_numpy()
view_rms = np.sqrt((views**2).mean(axis=1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

sc0 = axes[0].scatter(phi[:, 0], phi[:, 1], c=view_years, s=8, alpha=0.7, cmap="viridis")
axes[0].set_xlabel("φ₁  (spectral coordinate 1)"); axes[0].set_ylabel("φ₂  (spectral coordinate 2)")
axes[0].set_title("colored by calendar year")
plt.colorbar(sc0, ax=axes[0], label="year")

sc1 = axes[1].scatter(phi[:, 0], phi[:, 1], c=view_rms, s=8, alpha=0.7,
                       cmap="plasma", norm=LogNorm())
axes[1].set_xlabel("φ₁  (spectral coordinate 1)"); axes[1].set_ylabel("φ₂  (spectral coordinate 2)")
axes[1].set_title("colored by view RMS (residual amplitude, log)")
plt.colorbar(sc1, ax=axes[1], label="view RMS")

fig.suptitle(
    "Spectral embedding of 20-day HAR-residual windows  (each point = one trading day)\n"
    "Finding: years are intermingled (left) but RMS is a smooth gradient (right) — "
    "φ₁ essentially encodes volatility *level*, not regime.",
    fontsize=10,
)
plt.tight_layout(); plt.show()

In [ ]:
WINDOWS = [
    ("GFC",          "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt",      "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue","2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon",  "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID",        "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23",  "2023-03-08", "2023-03-22", "tab:green"),
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(phi[:, 0], phi[:, 1], c="lightgray", s=6, alpha=0.5, label="all views")
for label, lo, hi, color in WINDOWS:
    mask = (view_dates >= pd.Timestamp(lo)) & (view_dates < pd.Timestamp(hi))
    if mask.sum() == 0:
        continue
    ax.scatter(phi[mask, 0], phi[mask, 1], c=color, s=24, alpha=0.95,
               edgecolors="k", linewidths=0.4, label=f"{label}  (n={mask.sum()})")
ax.set_xlabel("φ₁  (spectral coordinate 1)"); ax.set_ylabel("φ₂  (spectral coordinate 2)")
fig.suptitle(
    "Where do known crises land in the embedding?\n"
    "If regimes were captured, each episode would form a tight, separated blob. "
    "Instead they smear along the high-amplitude tail — distinct crises are not separated.",
    fontsize=10,
)
ax.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

## 6. Higher embedding dimensions

`φ₁`–`φ₂` carry most of the structure, but the embedding has `d = 8` dims. The
pairs plot below checks whether `φ₃`/`φ₄` add anything beyond noise around
zero.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        if i <= j:
            ax.set_visible(False)
            continue
        ax.scatter(phi[:, j], phi[:, i + 1], c=view_years, cmap="viridis", s=4, alpha=0.6)
        ax.set_xlabel(f"φ_{j + 1}")
        ax.set_ylabel(f"φ_{i + 2}")
fig.suptitle(
    "First four embedding dimensions (lower-triangle pairs), colored by year\n"
    "Compact, near-symmetric clouds with no year separation — the useful "
    "dimensionality is low and still amplitude-driven.",
    y=1.01, fontsize=10,
)
plt.tight_layout(); plt.show()

## 7. Nyström extension — self-consistency check

The production backtest embeds a *new* view at every step via
`SpectralBasis.embed` (Nyström). As a first sanity check, re-embed 200
**training** views and compare to their true coordinates. This is a lower
bound on error (these points were in the index) — a proper held-out test
follows in Section 9.

In [ ]:
n_check = 200
check_idx = np.arange(len(views) - n_check, len(views))
phi_recovered = basis.embed_batch(views[check_idx])
rmse = np.sqrt(((phi_recovered - phi[check_idx]) ** 2).mean(axis=1))
phi_scale = phi.std(axis=0).mean()

print(f"Nystrom-vs-true RMSE on last 200 training views:")
print(f"  median:   {np.median(rmse):.4f}")
print(f"  max:      {rmse.max():.4f}")
print(f"  vs embedding scale (mean std per dim): {phi_scale:.4f}")
print(f"  relative median:  {np.median(rmse) / phi_scale * 100:.1f}% of embedding scale")

## 8. V2 — remove amplitude: z-normalize each view

Section 5 showed φ₁ ≈ volatility *level*. V2 tests whether *shape* alone
carries regime information: z-normalize each view (subtract its mean, divide
by its std) before embedding, so the distance can no longer see amplitude.

If shape were regime-discriminative, V2 should reveal structure that V1 hid
behind amplitude.

In [ ]:
def znormalize_views(arr):
    """Per-view z-normalization: subtract row mean, divide by row std."""
    m = arr.mean(axis=1, keepdims=True)
    s = arr.std(axis=1, keepdims=True) + 1e-12
    return (arr - m) / s


views_z = znormalize_views(views)
%time basis_v2 = build_embedding(views_z, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)
phi_v2 = basis_v2.phi_train

print(f"V2 phi shape: {phi_v2.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for col, (label, phi_v) in enumerate([("V1: raw Euclidean", phi), ("V2: z-normalized (shape-only)", phi_v2)]):
    sc0 = axes[0, col].scatter(phi_v[:, 0], phi_v[:, 1], c=view_years, s=6, alpha=0.7, cmap="viridis")
    axes[0, col].set_xlabel("φ₁"); axes[0, col].set_ylabel("φ₂")
    axes[0, col].set_title(f"{label} — by year")
    plt.colorbar(sc0, ax=axes[0, col], label="year")

    sc1 = axes[1, col].scatter(phi_v[:, 0], phi_v[:, 1], c=view_rms, s=6, alpha=0.7,
                                cmap="plasma", norm=LogNorm())
    axes[1, col].set_xlabel("φ₁"); axes[1, col].set_ylabel("φ₂")
    axes[1, col].set_title(f"{label} — by view RMS (log)")
    plt.colorbar(sc1, ax=axes[1, col], label="view RMS")

fig.suptitle(
    "V1 (raw) vs V2 (z-normalized) embeddings — rows: year (top), RMS (bottom)\n"
    "V2 removes the raw amplitude axis but still shows no year separation, and RMS still leaks "
    "(bursty windows share a shape) — shape alone does not separate regimes.",
    y=1.005, fontsize=10,
)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for ax, (label, phi_v) in zip(axes, [("V1: raw Euclidean", phi), ("V2: z-normalized", phi_v2)]):
    ax.scatter(phi_v[:, 0], phi_v[:, 1], c="lightgray", s=4, alpha=0.45, label="all views")
    for win_label, lo, hi, color in WINDOWS:
        mask = (view_dates >= pd.Timestamp(lo)) & (view_dates < pd.Timestamp(hi))
        if mask.sum() == 0:
            continue
        ax.scatter(phi_v[mask, 0], phi_v[mask, 1], c=color, s=22, alpha=0.95,
                   edgecolors="k", linewidths=0.3, label=f"{win_label} ({mask.sum()})")
    ax.set_xlabel("φ₁"); ax.set_ylabel("φ₂")
    ax.set_title(label)
axes[0].legend(fontsize=7, loc="best")
fig.suptitle(
    "Crisis windows under V1 vs V2\n"
    "Neither metric pulls the named crises into separated clusters — corroborates that "
    "single-channel residual windows (raw or shape) lack regime-discriminative structure.",
    y=1.02, fontsize=10,
)
plt.tight_layout(); plt.show()

## 9. Nyström out-of-sample backtest

The Section 7 check re-embedded *training* views (self-consistent by
construction). The honest test: hold out 10% of views, fit the embedding on
the rest, Nyström-extend the held-out views, and compare to where they land
when the embedding is fit on the full set. Spectral eigenvectors are defined
only up to rotation/sign, so we Procrustes-align the two frames via the shared
training points before measuring per-point error. We run it for both V1 and
V2 — this is the error the production `spectral_knn` backtest actually
incurs each step.

In [ ]:
from scipy.linalg import orthogonal_procrustes


def nystrom_holdout(views_arr, holdout_frac=0.10, seed=0):
    """Holdout-test Nystrom extension on `views_arr`.

    Returns the per-point alignment error, the embedding scale, and the
    points needed to plot truth-vs-extended.
    """
    rng_local = np.random.default_rng(seed)
    n = len(views_arr)
    test_i = rng_local.choice(n, size=int(holdout_frac * n), replace=False)
    train_i = np.setdiff1d(np.arange(n), test_i)

    b_train = build_embedding(views_arr[train_i], d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)
    b_full = build_embedding(views_arr, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)

    # Procrustes-align train_only -> full using the shared training points.
    A = b_full.phi_train[train_i]
    B = b_train.phi_train
    R, _ = orthogonal_procrustes(A, B)

    # Extend test views in train_only's frame, then rotate to full frame.
    phi_nystrom = b_train.embed_batch(views_arr[test_i]) @ R
    phi_truth = b_full.phi_train[test_i]

    err = np.linalg.norm(phi_nystrom - phi_truth, axis=1)
    scale = b_full.phi_train.std(axis=0).mean()

    return dict(
        test_idx=test_i, train_idx=train_i,
        phi_nystrom=phi_nystrom, phi_truth=phi_truth,
        phi_train_truth=A, err=err, scale=scale,
    )


print("Nystrom holdout backtest on V1 (raw Euclidean) ...")
out_v1 = nystrom_holdout(views)
print(f"  test views: {len(out_v1['test_idx'])}, train views: {len(out_v1['train_idx'])}")
print(f"  per-point alignment distance:")
print(f"    median  = {np.median(out_v1['err']):.4f}")
print(f"    90th    = {np.quantile(out_v1['err'], 0.9):.4f}")
print(f"    max     = {out_v1['err'].max():.4f}")
print(f"    vs embedding scale = {out_v1['scale']:.4f}")
print(f"    relative median = {np.median(out_v1['err']) / out_v1['scale'] * 100:.1f}% of scale")

In [ ]:
print("Nystrom holdout backtest on V2 (z-normalized) ...")
out_v2 = nystrom_holdout(views_z)
print(f"  test views: {len(out_v2['test_idx'])}, train views: {len(out_v2['train_idx'])}")
print(f"  per-point alignment distance:")
print(f"    median  = {np.median(out_v2['err']):.4f}")
print(f"    90th    = {np.quantile(out_v2['err'], 0.9):.4f}")
print(f"    max     = {out_v2['err'].max():.4f}")
print(f"    vs embedding scale = {out_v2['scale']:.4f}")
print(f"    relative median = {np.median(out_v2['err']) / out_v2['scale'] * 100:.1f}% of scale")

In [ ]:
# Side-by-side: V1 truth-vs-Nystrom and V2 truth-vs-Nystrom.
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for row, (label, out) in enumerate([("V1: raw Euclidean", out_v1), ("V2: z-normalized", out_v2)]):
    ax_truth = axes[row, 0]
    ax_nyst = axes[row, 1]

    ax_truth.scatter(out["phi_train_truth"][:, 0], out["phi_train_truth"][:, 1],
                      c="lightgray", s=4, alpha=0.4, label="train")
    ax_truth.scatter(out["phi_truth"][:, 0], out["phi_truth"][:, 1],
                      c="tab:blue", s=10, alpha=0.7, label="test (truth)")
    ax_truth.set_title(f"{label} -- full-set embedding")
    ax_truth.set_xlabel("phi_1"); ax_truth.set_ylabel("phi_2")
    ax_truth.legend(fontsize=8, loc="best")

    ax_nyst.scatter(out["phi_train_truth"][:, 0], out["phi_train_truth"][:, 1],
                     c="lightgray", s=4, alpha=0.4, label="train")
    ax_nyst.scatter(out["phi_nystrom"][:, 0], out["phi_nystrom"][:, 1],
                     c="tab:red", s=10, alpha=0.7, label="test (Nystrom)")
    ax_nyst.set_title(f"{label} -- train-only + Nystrom extension")
    ax_nyst.set_xlabel("phi_1"); ax_nyst.set_ylabel("phi_2")
    ax_nyst.legend(fontsize=8, loc="best")

plt.suptitle("Nystrom out-of-sample extension -- truth vs extended", y=1.005)
plt.tight_layout(); plt.show()

## Findings

1. **The embedding encodes volatility level, not regime.** φ₁ is a smooth,
   monotone function of view RMS (§5, right panel). Calendar years are
   intermingled and named crises smear along the amplitude tail rather than
   forming separated clusters (§5–6).

2. **Removing amplitude (V2) does not recover regimes.** Z-normalizing each
   view eliminates the raw-amplitude axis but yields no year/regime
   separation; RMS still leaks because bursty windows share a *shape* (§8).
   Conclusion: a single residual channel — raw or shape-normalized — lacks
   regime-discriminative structure over 20-day windows.

3. **Nyström extension is usable but imprecise on this data** (§9). Held-out
   per-point error is a sizeable fraction of the embedding scale, consistent
   with a noisy manifold rather than crisp clusters.

### Implications for the model

- The HAR-Ridge baseline already absorbs most regime-relevant variation, so
  its residual is residual *noise* more than residual *regime*.
- Regime identity is plausibly a **cross-channel** property (how RV residuals
  co-move with returns, sentiment, VIX), not a single-series shape — this
  motivates the correlation-matrix fingerprint of Papenbrock & Schwendner
  (2015) as the next variant (V4).
- A separate methodological caveat for the `spectral_knn` regressor: the
  k-NN matches on embedding distance but averages amplitude-laden residual
  targets, so a "better" (shape-aware) embedding can predict *worse* unless
  the target is scale-normalized.